# 04 · Reading and penalized head

This notebook applies the fine-tuned models of notebook 03 to every evaluation recording and stores their outputs. No decision is taken here:
thresholds, decision changes and all other outcomes are calculated from the stored outputs in notebook 05.

For both 5-s segments of every recording in `EVAL_RECORDS` it stores the logits of

- a **single read** of the segment, and the **25 window readings** of the segment (2.5-s windows; their mean probability is the window average of mixed reading);
- with the **model head** (the output layer trained with the network) and with the **penalized head** refitted with each penalty weight γ in `PENALTY_WEIGHTS`
  (0, 30, 100, 300, 1,000; 0 is a head refitted without a penalty).

Which outputs are needed depends on the model:

| Models | Outputs | Used for |
|---|---|---|
| baseline and student of the three foundation models (seeds 0–2) | single read and window readings, model head and all penalized heads | main analysis, effect of each remedy, trade-off of the penalized head |
| ECGFounder: `windows`, `agreement`, `selfdistill` (seeds 0–2) | single read, model head | alternatives for the training remedy |
| ECGFounder: baseline seeds 3–9 | single read, model head | ensemble of ten baseline models |
| ECGFounder: `dropout10` … `dropout50` (seeds 0–2) | mean of 25 passes with dropout, PTB-XL only | Monte Carlo dropout |

Teachers are not read: a teacher is needed only to calculate the distillation targets.

**Output files**

- `LOGITS/{tag}.npz`: float16 logits, rows in the order of `EVAL_RECORDS`, second axis = the two segments, last axis = the five classes in the order of `CLASSES`.
  `model_single` (n, 2, 5), `model_windows` (n, 2, 25, 5), `pen{γ}_single`, `pen{γ}_windows`, and `record`, `set` as a check of the order of the rows.
- `PENALIZED_HEADS/{tag}.npz`: the fitted heads (`gammas`, `mean`, `sd`, `W`, `b`).
- `MC_DROPOUT/{tag}.npz`: `mc` (m, 2, 5) probabilities and `row` (rows of `EVAL_RECORDS`).

**Time** on one RTX 3080: for a model with all outputs about 6 min (ECGFounder), 30 min (HuBERT-ECG) and 22 min (ECG-FM), of which the fitting of the heads takes 0.5 to 2 min;
less than 1 min for a model with a single read only, and about 1 min per model for Monte Carlo dropout. About 6 h in total. Every part skips the files that exist.

In [ ]:
%run ./00_setup_and_settings.ipynb

In [ ]:
QUICK_TEST = False        # True: functional check with a few recordings, using the models that notebook 03 wrote in its own QUICK_TEST mode; outputs go to subfolders "quick_test"

SUB = "quick_test" if QUICK_TEST else ""
MODEL_DIR, LOGITS_DIR, HEAD_DIR, MC_DIR = FINETUNED / SUB, LOGITS / SUB, PENALIZED_HEADS / SUB, MC_DROPOUT / SUB
for _d in (LOGITS_DIR, HEAD_DIR, MC_DIR):
    _d.mkdir(parents=True, exist_ok=True)
assert DEVICE.type == "cuda", "this notebook needs a GPU"
print("GPU:", torch.cuda.get_device_name(0), "| models from", MODEL_DIR)

## Evaluation recordings

`EVAL_RECORDS` has one row for every evaluation recording of every set (notebook 02). A recording that belongs to two sets has two rows, and it is simply read twice.
The order of the rows is the order of the rows of every output file.

In [ ]:
EVAL = pd.read_csv(EVAL_RECORDS)
EVAL["eval_row"] = np.arange(len(EVAL))
if QUICK_TEST:            # the first 40 single recordings and the first 20 pairs of every set
    EVAL = EVAL[((EVAL.pair_id < 0) & (EVAL.groupby("set").cumcount() < 40)) | ((EVAL.pair_id >= 0) & (EVAL.pair_id < 20))].reset_index(drop=True)
print(f"{len(EVAL):,} rows")
print(EVAL.groupby(["dataset", "set"]).size().to_string())

## Models to read

In [ ]:
# all outputs: single read and window readings, model head and penalized heads
FULL_JOBS = [(model, kind, seed) for model in MODELS for kind in ("baseline", "student") for seed in SEEDS]
# single read with the model head only: the alternatives for the training remedy and the further baseline models of the ensemble (ECGFounder)
SINGLE_JOBS = [("ecgfounder", kind, seed) for kind in ("windows", "agreement", "selfdistill") for seed in SEEDS]
SINGLE_JOBS += [("ecgfounder", "baseline", seed) for seed in ENSEMBLE_SEEDS if seed not in SEEDS]
# Monte Carlo dropout
DROPOUT_JOBS = [("ecgfounder", f"dropout{round(100 * rate)}", seed) for rate in DROPOUT_RATES for seed in SEEDS]

if QUICK_TEST:            # the models that notebook 03 trains in its QUICK_TEST mode
    FULL_JOBS = [j for j in FULL_JOBS if j[2] == 0]
    SINGLE_JOBS = [j for j in SINGLE_JOBS if j[2] == 0 and j[1] != "baseline"]
    DROPOUT_JOBS = DROPOUT_JOBS[:1]
print("models with all outputs:", len(FULL_JOBS), "| with a single read only:", len(SINGLE_JOBS), "| for Monte Carlo dropout:", len(DROPOUT_JOBS))

## Single read and window readings

`read_table` reads both segments of each recording once, and each of their 25 windows once. The network gives an embedding for every input, and every head is a linear function of it,
so all heads are obtained from the same forward passes: 2 + 2 × 25 = 52 passes for a recording.

A window reading places 25 windows of 2.5 s in the segment with evenly spaced onsets, the first at the start of the segment and the last ending at its end (`READ_STARTS`, notebook 00).
Every window lies within the segment, so the readings use no signal outside the 5 s that the single read uses.
Each input is scaled with its own samples inside the model class (notebook 00).

In [ ]:
@torch.no_grad()
def read_table(net, table, heads=None, windows=True, batch_records=32, verbose=True):
    """Logits for both segments of the recordings of `table` (columns `array`, `row`), as a dictionary of float16 arrays:
    "{head}_single" (n, 2, 5) and, if windows, "{head}_windows" (n, 2, 25, 5); head = "model" and, if heads are given, "pen{gamma}" for each fitted penalty weight."""
    n, t0 = len(table), time.time()
    names = ["model"] + ([f"pen{g}" for g in heads["gammas"]] if heads is not None else [])
    out = {f"{h}_single": np.zeros((n, 2, len(CLASSES)), np.float16) for h in names}
    if windows:
        out.update({f"{h}_windows": np.zeros((n, 2, N_WIN_READ, len(CLASSES)), np.float16) for h in names})
    W0, b0 = net.head.weight.detach().float().cpu().numpy().T, net.head.bias.detach().float().cpu().numpy()

    def all_heads(E):                             # embeddings (m, D) -> {head: logits (m, 5)}
        z = {"model": E @ W0 + b0}                # the model head takes the embedding as it is; the penalized heads take the standardized embedding
        if heads is not None:
            z.update({f"pen{g}": head_logits(E, heads, g) for g in heads["gammas"]})
        return z

    for a in range(0, n, batch_records):
        x = two_segments(torch.from_numpy(read_records(table.iloc[a:a + batch_records])).to(DEVICE)).flatten(0, 1)      # (B * 2, 12, 2500): segment 1 and 2 of each recording
        B = len(x) // 2
        for h, z in all_heads(embed_inputs(net, x)).items():
            out[f"{h}_single"][a:a + B] = z.reshape(B, 2, -1)
        if windows:
            w = cut_pieces(x, READ_STARTS, WIN).flatten(0, 1)                                                           # (B * 2 * 25, 12, 1250)
            for h, z in all_heads(embed_inputs(net, w)).items():
                out[f"{h}_windows"][a:a + B] = z.reshape(B, 2, N_WIN_READ, -1)
        if verbose and (a // batch_records) % 400 == 0:
            print(f"    {a:,} / {n:,} recordings | {(time.time() - t0) / 60:.1f} min", flush=True)
    return out

## Penalized head

The heads are fitted with `fit_penalized_heads` (notebook 00, section *Penalized head*), with the network fixed, on the two segments of the 17,084 PTB-XL training recordings (34,168 examples):

1. embeddings of every training segment and of its two 4.5-s views that start 0.5 s apart;
2. standardization of the embedding with the mean and SD over the training segments;
3. covariance S of the differences between the standardized embeddings of the two views, each difference divided by √2, scaled so that the trace of S equals the number of dimensions;
4. one logistic regression for each diagnostic class (C = `HEAD_C` = 0.01, balanced class weights) after the standardized embeddings are multiplied by (I + γS)^(−1/2),
   which is the same as adding the penalty γ wᵀSw to the ridge penalty on the weights w.

The evaluation recordings play no part in the fit. One fit is made for each γ in `PENALTY_WEIGHTS`; γ = 300 is the value of the final setting, and γ = 0 gives a head that is refitted without a penalty.
The head with γ = 0 differs from the model head only in how it was fitted: on the embeddings of the final network, with a ridge penalty, and with every training segment used once.

## Outputs of every model

`write_logits` does, for one model: load the model, fit the penalized heads (or load them if they were saved), read all evaluation recordings, write the file.
A model whose file exists is skipped.

In [ ]:
def write_logits(tag, full):
    """Read all recordings of EVAL with one fine-tuned model and write LOGITS_DIR / "{tag}.npz". full = window readings and penalized heads as well."""
    out_file = LOGITS_DIR / f"{tag}.npz"
    if out_file.exists():
        return
    if not (MODEL_DIR / f"{tag}.pt").exists():
        print(f"[{tag}] no model file in {MODEL_DIR}: skipped"); return
    t0 = time.time(); net = load_finetuned(tag, MODEL_DIR); heads = None
    if full:
        if (HEAD_DIR / f"{tag}.npz").exists():
            heads = load_heads(tag, HEAD_DIR)
        else:
            heads = fit_penalized_heads(net, n_train=640 if QUICK_TEST else None, verbose=False); save_heads(heads, tag, HEAD_DIR)
            print(f"[{tag}] {len(heads['gammas'])} heads fitted: embeddings {heads['seconds_embeddings'] / 60:.1f} min, fits {heads['seconds_fit'] / 60:.1f} min", flush=True)
    out = read_table(net, EVAL, heads, windows=full)
    tmp = LOGITS_DIR / f"{tag}.tmp.npz"                       # written under another name first, so that an interrupted run leaves no incomplete file
    np.savez(tmp, **out, record=EVAL.record.values, set=EVAL.set.values.astype(str)); tmp.replace(out_file)
    print(f"[{tag}] {len(out)} arrays ({'single read and window readings, model head and penalized heads' if full else 'single read, model head'}) | {(time.time() - t0) / 60:.1f} min", flush=True)
    del net; torch.cuda.empty_cache()

### Baseline and student of the three foundation models

Takes about 6 min for each ECGFounder model, 30 min for each HuBERT-ECG model and 22 min for each ECG-FM model (RTX 3080); each file has about 0.16 GB.

In [ ]:
for model, kind, seed in FULL_JOBS:                          # about 6 h on one RTX 3080 for the 18 models
    write_logits(model_tag(model, kind, seed), full=True)

### Comparison models of ECGFounder

The alternatives for the training remedy and the baseline models of the ensemble are read once with the model head (less than 1 min each).
The ensemble itself, the mean of the probabilities of the ten baseline models, is formed in notebook 05.

In [ ]:
for model, kind, seed in SINGLE_JOBS:                        # about 10 min for the 16 models
    write_logits(model_tag(model, kind, seed), full=False)

### Check of the files

A quick look at the outputs before the analysis: the files that exist, the number of arrays in each, and the mean AUROC of the five classes in the PTB-XL test fold
for a single read with the model head and with the penalized head of the final setting (mean of the two segments). The values of the paper come from notebook 05, not from here.

In [ ]:
def _auroc(logits, labels):
    try:
        return float(np.mean([roc_auc_score(labels[:, j], logits[:, h, j]) for j in range(len(CLASSES)) for h in (0, 1)]))
    except ValueError:                            # a class without positive or without negative recordings (possible in QUICK_TEST)
        return float("nan")


_te = (EVAL.set == "ptbxl_test").values; _y = EVAL.loc[_te, CLASSES].values.astype(int); _rows = []
for model, kind, seed in FULL_JOBS + SINGLE_JOBS:
    f = LOGITS_DIR / f"{model_tag(model, kind, seed)}.npz"
    if f.exists():
        z = np.load(f); assert (z["record"] == EVAL.record.values).all() and (z["set"] == EVAL.set.values.astype(str)).all(), f.name
        _rows.append({"model": f.stem, "arrays": len(z.files) - 2, "AUROC, model head": _auroc(z["model_single"][_te].astype(np.float32), _y),
                      "AUROC, penalized head": _auroc(z[f"pen{PENALTY_WEIGHT}_single"][_te].astype(np.float32), _y) if f"pen{PENALTY_WEIGHT}_single" in z.files else np.nan})
pd.DataFrame(_rows).round(4)

## Monte Carlo dropout

The comparison with Monte Carlo dropout is made in ECGFounder and in PTB-XL. The models are baselines trained with dropout after each activation in the residual blocks
(`dropout10` … `dropout50`, notebook 03). Each segment of the PTB-XL validation and test recordings is read 25 times with dropout on, with independent random masks,
and the probabilities of the 25 passes are averaged (`mc`).

The number of passes equals the number of windows of mixed reading. Thresholds, the choice of the dropout rate by the validation AUROC, and the outcomes are calculated in notebook 05.
The masks are random: the seed below fixes them for a given order of the recordings and a given batch size, but another graphics card or library version can give other masks.

In [ ]:
MC_PASSES = 25                                    # passes with dropout on
MC_SEED = 0x13528FB                               # seed of the dropout masks (a fixed number)


def set_dropout(net, on):
    """Evaluation mode for the whole model; with on=True the dropout layers alone are put back into training mode, so that they draw random masks."""
    net.eval()
    if on:
        for m in net.modules():
            if isinstance(m, nn.Dropout):
                m.train()


@torch.no_grad()
def mc_dropout_read(net, table, passes=MC_PASSES, batch_records=256):
    """Probabilities (m, 2, 5) for both segments of the recordings of `table`: the mean over `passes` reads with dropout on."""
    m = len(table); mc = np.zeros((m, 2, len(CLASSES)))
    set_dropout(net, True)
    for a in range(0, m, batch_records):
        x = two_segments(torch.from_numpy(read_records(table.iloc[a:a + batch_records])).to(DEVICE)).flatten(0, 1)

        def prob():
            return torch.sigmoid(model_logits(net, x)).cpu().numpy().astype(np.float64).reshape(-1, 2, len(CLASSES))

        mc[a:a + len(x) // 2] = np.mean([prob() for _ in range(passes)], 0)
    set_dropout(net, False)
    return mc

In [ ]:
PTB = EVAL[EVAL.set.isin(["ptbxl_val", "ptbxl_test"])]
if QUICK_TEST:
    PTB = PTB.iloc[:64]
for model, kind, seed in DROPOUT_JOBS:                       # about 15 min for the 15 models
    tag = model_tag(model, kind, seed); out_file = MC_DIR / f"{tag}.npz"
    if out_file.exists():
        continue
    if not (MODEL_DIR / f"{tag}.pt").exists():
        print(f"[{tag}] no model file in {MODEL_DIR}: skipped"); continue
    t0 = time.time(); net = load_finetuned(tag, MODEL_DIR)
    torch.manual_seed(MC_SEED)
    mc = mc_dropout_read(net, PTB)
    tmp = MC_DIR / f"{tag}.tmp.npz"; np.savez(tmp, mc=mc.astype(np.float32), row=PTB.eval_row.values); tmp.replace(out_file)
    print(f"[{tag}] dropout rate {kind_settings(model, kind)['dropout']}: {len(PTB):,} recordings x {MC_PASSES} passes | {(time.time() - t0) / 60:.1f} min", flush=True)
    del net; torch.cuda.empty_cache()

## Files written

In [ ]:
for _name, _dir in (("logits", LOGITS_DIR), ("penalized heads", HEAD_DIR), ("Monte Carlo dropout", MC_DIR)):
    _files = sorted(_dir.glob("*.npz"))
    print(f"{_name}: {len(_files)} files, {sum(f.stat().st_size for f in _files) / 2 ** 30:.2f} GB in {_dir}")